# 🚗 Vehicle Maintenance Risk Prediction — EDA & Model Training

**Project:** ML Capstone – Milestone 1  
**Objective:** Predict whether a vehicle is at HIGH RISK of failure using classical ML.  
**Target:** `Failure_Probability` (Binary Classification: 0 = No risk, 1 = Risk)  
**Models:** Logistic Regression, Decision Tree Classifier  
**Framework:** Scikit-learn

## 1. Setup & Imports

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ All imports successful")

✅ All imports successful


## 2. Load Dataset

In [12]:
# Load from local file (use relative path from notebooks/ folder)
df = pd.read_csv("../data/EV_Predictive_Maintenance_Dataset.csv")

print(f"Dataset shape: {df.shape}")
print(f"Records: {df.shape[0]:,}  |  Features: {df.shape[1]}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/EV_Predictive_Maintenance_Dataset.csv'

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Missing": missing, "Percent": missing_pct})
print(missing_df[missing_df["Missing"] > 0] if missing.sum() > 0 else "✅ No missing values found")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Failure_Probability'].value_counts().plot.bar(ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Failure Probability Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No Failure (0)', 'Failure (1)'], rotation=0)

df['Failure_Probability'].value_counts().plot.pie(
    ax=axes[1], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
    labels=['No Failure', 'Failure']
)
axes[1].set_ylabel('')
axes[1].set_title('Failure Probability Split')

plt.tight_layout()
plt.show()

print(f"\nClass distribution:\n{df['Failure_Probability'].value_counts()}")
print(f"\nClass balance: {df['Failure_Probability'].value_counts(normalize=True).round(3).to_dict()}")

In [ ]:
# Distribution of key numeric features
key_features = [
    'SoC', 'SoH', 'Battery_Voltage', 'Battery_Temperature',
    'Motor_Temperature', 'Motor_RPM', 'Brake_Pad_Wear',
    'Driving_Speed', 'Distance_Traveled'
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for i, feat in enumerate(key_features):
    ax = axes[i // 3, i % 3]
    df[feat].hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(feat)
    ax.set_xlabel('')

plt.suptitle('Distribution of Key Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (exclude leakage columns)
cols_for_corr = [c for c in df.select_dtypes(include=[np.number]).columns
                 if c not in ['RUL', 'TTF', 'Component_Health_Score', 'Timestamp']]
corr = df[cols_for_corr].corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots: key features by Failure_Probability
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
box_feats = ['Battery_Temperature', 'Motor_Temperature', 'Brake_Pad_Wear',
             'Driving_Speed', 'SoH', 'Motor_RPM']

for i, feat in enumerate(box_feats):
    ax = axes[i // 3, i % 3]
    df.boxplot(column=feat, by='Failure_Probability', ax=ax)
    ax.set_title(feat)
    ax.set_xlabel('Failure Probability')

plt.suptitle('Feature Distributions by Failure Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

### 4.1 Remove Data Leakage Columns
The following columns **directly encode failure information** and must be removed to prevent data leakage:
- `RUL` (Remaining Useful Life)
- `TTF` (Time to Failure)
- `Component_Health_Score`

In [ ]:
# Remove leakage columns
leakage_cols = ['RUL', 'TTF', 'Component_Health_Score']
df_clean = df.drop(columns=leakage_cols)

# Drop Timestamp (non-feature)
df_clean = df_clean.drop(columns=['Timestamp'])

print(f"Dropped leakage columns: {leakage_cols}")
print(f"Dropped non-feature columns: ['Timestamp']")
print(f"Remaining shape: {df_clean.shape}")

### 4.2 Feature Engineering

Create derived features:
- **Battery_Stress** = `Battery_Current × Battery_Temperature` — High current draw under high temperature stresses the battery
- **Motor_Load** = `Motor_Torque × Motor_RPM` — Represents total mechanical power output
- **Wear_Index** = `Distance_Traveled / Charge_Cycles` — Average distance per charge cycle; higher values indicate heavier usage patterns

In [ ]:
# Feature Engineering
df_clean['Battery_Stress'] = df_clean['Battery_Current'] * df_clean['Battery_Temperature']
df_clean['Motor_Load'] = df_clean['Motor_Torque'] * df_clean['Motor_RPM']
df_clean['Wear_Index'] = df_clean['Distance_Traveled'] / df_clean['Charge_Cycles'].replace(0, np.nan)

print("✅ Engineered features created:")
print(f"   Battery_Stress stats: mean={df_clean['Battery_Stress'].mean():.2f}, std={df_clean['Battery_Stress'].std():.2f}")
print(f"   Motor_Load stats:     mean={df_clean['Motor_Load'].mean():.2f}, std={df_clean['Motor_Load'].std():.2f}")
print(f"   Wear_Index stats:     mean={df_clean['Wear_Index'].mean():.2f}, std={df_clean['Wear_Index'].std():.2f}")
print(f"\nNew shape: {df_clean.shape}")

### 4.3 Handle Missing Values & Split Data

In [ ]:
# Handle missing values with median imputation
from sklearn.impute import SimpleImputer

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
imputer = SimpleImputer(strategy='median')
df_clean[numeric_cols] = imputer.fit_transform(df_clean[numeric_cols])

print(f"Missing values after imputation: {df_clean.isnull().sum().sum()}")

# Separate features and target
TARGET = 'Failure_Probability'
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

feature_names = X.columns.tolist()
print(f"\nFeatures ({len(feature_names)}): {feature_names}")
print(f"Target distribution:\n{y.value_counts()}")

In [ ]:
# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]:,} samples")
print(f"Test set:  {X_test.shape[0]:,} samples")
print(f"\nTrain target distribution:\n{y_train.value_counts()}")
print(f"\nTest target distribution:\n{y_test.value_counts()}")

In [ ]:
# Feature scaling with StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Feature scaling applied (StandardScaler)")
print(f"   Scaled train shape: {X_train_scaled.shape}")
print(f"   Scaled test shape:  {X_test_scaled.shape}")

## 5. Model Training

### 5.1 Logistic Regression
Logistic Regression is a linear model suitable for binary classification. It models the probability of failure using a logistic (sigmoid) function. Feature scaling is **required** because it uses gradient-based optimization.

In [ ]:
# Train Logistic Regression (with balanced class weights to handle imbalance)
lr_model = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs', class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test_scaled)
y_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Metrics
print("=" * 50)
print("LOGISTIC REGRESSION — Results")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_lr):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_proba_lr):.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred_lr)}")

In [ ]:
# Logistic Regression — Confusion Matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, ax=ax, cmap='Blues')
ax.set_title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Logistic Regression — Cross Validation
lr_cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='f1', n_jobs=-1)
print("Logistic Regression — 5-Fold Cross Validation (F1)")
print(f"   Scores: {lr_cv_scores}")
print(f"   Mean:   {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}")

### 5.2 Decision Tree Classifier (with Hyperparameter Tuning)
Decision Trees are non-linear models that split features hierarchically. They don't require feature scaling but are prone to overfitting. We use **GridSearchCV** to find optimal hyperparameters.

In [ ]:
# Hyperparameter tuning with GridSearchCV (balanced class weights)
param_grid = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')
grid_search = GridSearchCV(dt, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train_scaled, y_train)

dt_model = grid_search.best_estimator_
print(f"\n✅ Best parameters: {grid_search.best_params_}")
print(f"   Best CV F1 score: {grid_search.best_score_:.4f}")

In [ ]:
# Decision Tree — Predictions & Metrics
y_pred_dt = dt_model.predict(X_test_scaled)
y_proba_dt = dt_model.predict_proba(X_test_scaled)[:, 1]

print("=" * 50)
print("DECISION TREE — Results")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_dt):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_proba_dt):.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred_dt)}")

In [ ]:
# Decision Tree — Confusion Matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_dt, ax=ax, cmap='Greens')
ax.set_title('Decision Tree — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Decision Tree — Cross Validation
dt_cv_scores = cross_val_score(dt_model, X_train_scaled, y_train, cv=5, scoring='f1', n_jobs=-1)
print("Decision Tree — 5-Fold Cross Validation (F1)")
print(f"   Scores: {dt_cv_scores}")
print(f"   Mean:   {dt_cv_scores.mean():.4f} ± {dt_cv_scores.std():.4f}")

## 6. Model Comparison & ROC Curve

In [ ]:
# Model Comparison Table
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC AUC', 'CV F1 Mean'],
    'Logistic Regression': [
        accuracy_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr),
        roc_auc_score(y_test, y_proba_lr),
        lr_cv_scores.mean()
    ],
    'Decision Tree': [
        accuracy_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_dt),
        roc_auc_score(y_test, y_proba_dt),
        dt_cv_scores.mean()
    ]
})

print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
comparison

In [ ]:
# ROC Curve — Both Models
fig, ax = plt.subplots(figsize=(8, 6))

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_proba_dt)

ax.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, y_proba_lr):.4f})', linewidth=2)
ax.plot(fpr_dt, tpr_dt, label=f'Decision Tree (AUC = {roc_auc_score(y_test, y_proba_dt):.4f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Model Comparison')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Feature Importance (Decision Tree)

In [ ]:
# Feature Importance — Decision Tree (Top 15)
importances = dt_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 7))
top_n = 15
top_names = [feature_names[i] for i in sorted_idx[:top_n]]
top_vals = importances[sorted_idx[:top_n]]

bars = ax.barh(top_names[::-1], top_vals[::-1], color='teal', edgecolor='white')
ax.set_xlabel('Importance Score')
ax.set_title(f'Top {top_n} Feature Importances (Decision Tree)')
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for bar, val in zip(bars, top_vals[::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Print all importances
print("\nAll Feature Importances (sorted):")
for i, idx in enumerate(sorted_idx):
    print(f"   {i+1:2d}. {feature_names[idx]:30s} : {importances[idx]:.6f}")

## 8. Save Model Artefacts

Save the best-performing model, scaler, and feature names as `model.pkl` for use in the Streamlit app.

In [ ]:
# Determine the best model by F1 score
lr_f1 = f1_score(y_test, y_pred_lr)
dt_f1 = f1_score(y_test, y_pred_dt)

if dt_f1 >= lr_f1:
    best_model = dt_model
    best_name = "Decision Tree"
else:
    best_model = lr_model
    best_name = "Logistic Regression"

print(f"Best model: {best_name} (F1 = {max(lr_f1, dt_f1):.4f})")

# Build artefacts dict
artefacts = {
    "best_model": best_model,
    "best_model_name": best_name,
    "lr_model": lr_model,
    "dt_model": dt_model,
    "scaler": scaler,
    "feature_names": feature_names,
    "lr_metrics": {
        "accuracy": accuracy_score(y_test, y_pred_lr),
        "precision": precision_score(y_test, y_pred_lr),
        "recall": recall_score(y_test, y_pred_lr),
        "f1": f1_score(y_test, y_pred_lr),
        "roc_auc": roc_auc_score(y_test, y_proba_lr),
        "confusion_matrix": confusion_matrix(y_test, y_pred_lr),
        "classification_report": classification_report(y_test, y_pred_lr),
    },
    "dt_metrics": {
        "accuracy": accuracy_score(y_test, y_pred_dt),
        "precision": precision_score(y_test, y_pred_dt),
        "recall": recall_score(y_test, y_pred_dt),
        "f1": f1_score(y_test, y_pred_dt),
        "roc_auc": roc_auc_score(y_test, y_proba_dt),
        "confusion_matrix": confusion_matrix(y_test, y_pred_dt),
        "classification_report": classification_report(y_test, y_pred_dt),
    },
}

# Save to project root
model_path = "../model.pkl"
joblib.dump(artefacts, model_path)
print(f"✅ Artefacts saved to {model_path}")

## 9. Conclusion

### Key Findings:
1. **Data Leakage Prevention**: Removed `RUL`, `TTF`, and `Component_Health_Score` to ensure the model learns from actual telemetry features only.
2. **Feature Engineering**: Created `Battery_Stress`, `Motor_Load`, and `Wear_Index` — composite features capturing real-world degradation patterns.
3. **Both models** (Logistic Regression and Decision Tree) were trained, evaluated, and compared.
4. **Cross-validation** confirms generalization and reduces overfitting risk.
5. **Hyperparameter tuning** (GridSearchCV) was applied to the Decision Tree.

### Why Logistic Regression?
- Simple, interpretable baseline for binary classification
- Works well when the decision boundary is approximately linear
- Requires feature scaling (StandardScaler applied)

### Why Decision Tree?
- Captures non-linear relationships and feature interactions
- Provides feature importance rankings
- Does not assume any data distribution

### Why was scaling required?
- Logistic Regression uses gradient-based optimization (LBFGS) — features on different scales cause slow convergence and biased coefficients

### What is data leakage?
- When information from the target (or its direct proxy) leaks into features during training, the model learns shortcuts instead of real patterns. RUL, TTF, and Component Health Score directly encode failure state.

The best-performing model has been saved as `model.pkl` for deployment via the Streamlit app.